# H2 — ¿Conviene considerar `ai_maturity_score`?

**Pregunta:** ¿incorporar madurez mejora la predicción de `ai_adoption_rate` en
empresas no utilizadas para ajustar el modelo, después de controlar industria,
tamaño, país, año y trimestre?

**H₀:** agregar madurez no reduce el error absoluto medio por empresa.
**H₁:** agregar madurez reduce ese error.

Definimos Δ = MAE del modelo base − MAE del modelo con madurez. Un Δ positivo
favorece incluir madurez desde el punto de vista predictivo. Informamos su magnitud
en puntos porcentuales de adopción y un intervalo bootstrap del 95% por empresa.
Un intervalo completamente positivo aporta evidencia a favor de H₁ en esta
evaluación exploratoria. No fijamos un umbral arbitrario de utilidad práctica:
mostramos también la reducción relativa del error.

Esta prueba evalúa utilidad predictiva. No determina si la fórmula de madurez
incluye adopción ni si el índice estaría disponible al momento de predecir.
Tampoco convierte asociaciones observacionales en relaciones causales.

## Cómo ejecutar este notebook

Cloná o descargá el repositorio completo, conservando su estructura. Este análisis usa únicamente `data/raw/ai_company_adoption.csv`, incluido en el repositorio: no requiere ejecutar `curacion.ipynb` ni tener los CSV derivados de otros prácticos.

En el entorno de Python que uses para Jupyter, instalá las dependencias:

```bash
python -m pip install numpy pandas scikit-learn ipykernel jupyterlab
```

Abrí este archivo en JupyterLab o en VS Code con soporte para notebooks, seleccioná ese entorno y ejecutá todas las celdas en orden. Puede iniciarse desde la raíz del repositorio o desde la carpeta `notebook`, en Windows, macOS o Linux. La ubicación se detecta a partir del CSV de entrada, sin rutas personales ni necesidad de la carpeta `.git`.

La ejecución se validó con Python 3.12.13, NumPy 2.5.2, pandas 3.0.5 y scikit-learn 1.9.0. La primera celda muestra las versiones utilizadas. Las semillas de la partición y del remuestreo están fijadas; los resultados numéricos pueden variar ligeramente entre versiones.

GitHub permite consultar el notebook y sus resultados guardados; para volver a calcularlos hace falta ejecutarlo en un entorno de Python. No se escriben ni se modifican archivos de datos.


In [1]:
from pathlib import Path
import hashlib
import sys
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display, Markdown

def encontrar_proyecto(inicio=None):
    """Busca los datos en la carpeta de trabajo y sus directorios superiores."""
    carpeta = Path.cwd() if inicio is None else Path(inicio)
    carpeta = carpeta.resolve()
    for candidata in (carpeta, *carpeta.parents):
        if (candidata / 'data' / 'raw' / 'ai_company_adoption.csv').is_file():
            return candidata
    raise FileNotFoundError(
        'No se encontró data/raw/ai_company_adoption.csv. '
        'Cloná o descargá el repositorio completo y ejecutá el notebook '
        'desde la raíz del proyecto o desde su carpeta notebook.'
    )

PROYECTO = encontrar_proyecto()
DATOS = PROYECTO / 'data' / 'raw'
TARGET = 'ai_adoption_rate'
SCORE = 'ai_maturity_score'
CONTROLES = ['industry', 'company_size', 'country', 'survey_year', 'quarter']
COMPONENTES = ['ai_training_hours', 'ai_budget_percentage', 'ai_projects_active']
SEED = 20260903
# El análisis solo lee este CSV; no depende de otros notebooks ni de datos curados.
protegidos = [DATOS / 'ai_company_adoption.csv']
def sha256(p):
    return hashlib.sha256(p.read_bytes()).hexdigest()
huellas = {str(p): sha256(p) for p in protegidos}

original = pd.read_csv(DATOS / 'ai_company_adoption.csv',
                       usecols=['response_id', 'company_id', TARGET, SCORE, *CONTROLES, *COMPONENTES],
                       float_precision='round_trip')
print(f'Python {sys.version.split()[0]} | pandas {pd.__version__} | sklearn {sklearn.__version__}')

Python 3.12.13 | pandas 3.0.5 | sklearn 1.9.0


## 1. Misma muestra para todos los modelos

Usamos el CSV original, evitando las imputaciones realizadas sobre toda la muestra
en la curación. Excluimos filas sin identificador, target, madurez o controles,
tasas fuera de 0–100, madurez fuera de 0–1 y períodos inválidos.

La comparación utiliza exactamente las mismas filas. Los faltantes de capacitación,
presupuesto y proyectos se imputan exclusivamente con la mediana del entrenamiento
en cada ajuste. Los valores negativos de conteos/horas y los presupuestos fuera
de 0–100 se marcan como ausentes mediante reglas de dominio, sin aprender umbrales
del test. No aplicamos aquí la exclusión exploratoria de horas superiores a 80.

Año y trimestre se codifican como categorías, evitando imponer una tendencia
lineal. `company_id` se utiliza únicamente para agrupar: nunca entra en X.
Se excluyen etapa de adopción, promedios sectoriales y variables de resultado.

In [2]:
requeridas = ['company_id', TARGET, SCORE, *CONTROLES]
completas = original.dropna(subset=requeridas).copy()
validas = (
    np.isfinite(completas[[TARGET, SCORE, 'survey_year']]).all(axis=1)
    & completas[TARGET].between(0, 100)
    & completas[SCORE].between(0, 1)
    & completas['survey_year'].mod(1).eq(0)
    & completas['quarter'].isin(['Q1', 'Q2', 'Q3', 'Q4'])
)
datos = completas.loc[validas].copy().reset_index(drop=True)
datos['survey_year'] = datos['survey_year'].astype(int).astype(str)
reglas_componentes = {
    'ai_training_hours': (0, None),
    'ai_budget_percentage': (0, 100),
    'ai_projects_active': (0, None),
}
for columna, (lo, hi) in reglas_componentes.items():
    fuera = ~np.isfinite(datos[columna]) | datos[columna].lt(lo)
    if hi is not None:
        fuera |= datos[columna].gt(hi)
    datos.loc[fuera, columna] = np.nan

display(pd.DataFrame([
    {'Paso': 'CSV original', 'Filas': len(original)},
    {'Paso': 'Completa en target, score, ID y controles', 'Filas': len(completas)},
    {'Paso': 'Muestra común con dominio válido', 'Filas': len(datos)},
]))
display(datos[COMPONENTES].isna().sum().to_frame('Faltantes a imputar en entrenamiento'))

,Paso,Filas
0,CSV original,150025
1,"Completa en target, score, ID y controles",143366
2,Muestra común con dominio válido,143072


,Faltantes a imputar en entrenamiento
ai_training_hours,2289
ai_budget_percentage,2152
ai_projects_active,0


## 2. Modelos comparados y validación

| Modelo | Variables |
|---|---|
| Base | Industria, tamaño, país, año y trimestre |
| Base + madurez | Controles y `ai_maturity_score` |
| Base + componentes | Controles, capacitación, presupuesto y proyectos |
| Base + componentes + madurez | Controles, los componentes y el índice |

La comparación principal es Base contra Base + madurez. Las otras dos son
diagnósticos secundarios de redundancia: ¿madurez agrega información cuando ya
están disponibles sus posibles componentes?

Usamos regresión lineal con las mismas decisiones de preparación y sin ajustar
hiperparámetros. Reservamos el 20% de las empresas para prueba final. Sobre el 80%
restante hacemos validación cruzada de cinco particiones por empresa. Los modelos
comparten las mismas particiones. Encoder, imputador y escalador se ajustan siempre
solo con el entrenamiento correspondiente.

La métrica principal promedia primero el error de cada empresa y luego entre
empresas, para que las que tienen más encuestas no pesen más. MAE, RMSE y R² por
fila se muestran como medidas complementarias. No recortamos las predicciones
lineales a 0–100.

In [3]:
ESPECIFICACIONES = {
    'Base': [],
    'Base + madurez': [SCORE],
    'Base + componentes': COMPONENTES,
    'Base + componentes + madurez': [*COMPONENTES, SCORE],
}
def construir_modelo(numericas):
    ramas = [('categoricas', OneHotEncoder(handle_unknown='ignore', drop='first'), CONTROLES)]
    if numericas:
        ramas.append(('numericas', Pipeline([
            ('imputar', SimpleImputer(strategy='median')),
            ('escalar', StandardScaler()),
        ]), numericas))
    return Pipeline([
        ('preparar', ColumnTransformer(ramas)),
        ('regresion', LinearRegression()),
    ])

def errores_empresa(y, pred, grupos):
    return pd.DataFrame({'empresa': np.asarray(grupos),
                         'error': np.abs(np.asarray(y) - pred)}).groupby('empresa')['error'].mean()

def metricas(y, pred, grupos):
    return {'MAE por empresa': errores_empresa(y, pred, grupos).mean(),
            'MAE por fila': mean_absolute_error(y, pred),
            'RMSE por fila': np.sqrt(mean_squared_error(y, pred)),
            'R2 por fila': r2_score(y, pred)}

idx_desarrollo, idx_test = next(GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
                               .split(datos, groups=datos['company_id']))
desarrollo = datos.iloc[idx_desarrollo].copy()
test = datos.iloc[idx_test].copy()
assert set(desarrollo.company_id).isdisjoint(set(test.company_id))
assert len(desarrollo) + len(test) == len(datos)
display(pd.DataFrame([
    {'Conjunto': 'Desarrollo / validación cruzada', 'Filas': len(desarrollo), 'Empresas': desarrollo.company_id.nunique()},
    {'Conjunto': 'Prueba final', 'Filas': len(test), 'Empresas': test.company_id.nunique()},
]))
for numericas in ESPECIFICACIONES.values():
    assert not {'company_id', 'response_id', 'ai_stage_enc', 'ai_adoption_stage', TARGET} & set(CONTROLES + numericas)

,Conjunto,Filas,Empresas
0,Desarrollo / validación cruzada,114417,8014
1,Prueba final,28655,2004


In [4]:
cv = GroupKFold(n_splits=5)
filas_cv = []
for fold, (itr, iva) in enumerate(cv.split(desarrollo, groups=desarrollo.company_id), start=1):
    train_fold, valid_fold = desarrollo.iloc[itr], desarrollo.iloc[iva]
    assert set(train_fold.company_id).isdisjoint(set(valid_fold.company_id))
    for nombre, numericas in ESPECIFICACIONES.items():
        features = CONTROLES + numericas
        modelo = construir_modelo(numericas)
        modelo.fit(train_fold[features], train_fold[TARGET])
        pred = modelo.predict(valid_fold[features])
        filas_cv.append({'Partición': fold, 'Modelo': nombre,
                         **metricas(valid_fold[TARGET], pred, valid_fold.company_id)})
    print(f'Partición {fold}/5 completada: cuatro modelos sobre las mismas empresas de validación.', flush=True)
resultados_cv = pd.DataFrame(filas_cv)
display(resultados_cv.pivot(index='Partición', columns='Modelo', values='MAE por empresa').round(5))
display(resultados_cv.groupby('Modelo')[['MAE por empresa', 'R2 por fila']].agg(['mean', 'std']).round(5))

Partición 1/5 completada: cuatro modelos sobre las mismas empresas de validación.


Partición 2/5 completada: cuatro modelos sobre las mismas empresas de validación.


Partición 3/5 completada: cuatro modelos sobre las mismas empresas de validación.


Partición 4/5 completada: cuatro modelos sobre las mismas empresas de validación.


Partición 5/5 completada: cuatro modelos sobre las mismas empresas de validación.


Modelo,Base,Base + componentes,Base + componentes + madurez,Base + madurez
Partición,,,,
1,10.99560,5.98782,5.86621,6.09935
2,10.92809,5.95257,5.85499,6.05831
3,11.01573,5.96458,5.84650,6.07766
4,10.88241,5.96844,5.84413,6.04923
5,10.94467,5.96727,5.87211,6.07945


MAE por empresa          R2 por fila         
                                        mean      std        mean      std
Modelo                                                                    
Base                                10.95330  0.05344     0.11586  0.00547
Base + componentes                   5.96813  0.01268     0.72887  0.00042
Base + componentes + madurez         5.85679  0.01217     0.74547  0.00236
Base + madurez                       6.07280  0.01961     0.72635  0.00214

## 3. Prueba final en empresas reservadas

Ajustamos cada modelo con todo el conjunto de desarrollo y evaluamos una sola vez
en las empresas reservadas. Este test es interno al mismo dataset, cuyos patrones
ya se exploraron en P1/P2; no equivale a validación externa ni temporal.

In [5]:
ajustes, predicciones, errores, filas_test = {}, {}, {}, []
for nombre, numericas in ESPECIFICACIONES.items():
    features = CONTROLES + numericas
    modelo = construir_modelo(numericas)
    modelo.fit(desarrollo[features], desarrollo[TARGET])
    pred = modelo.predict(test[features])
    ajustes[nombre] = modelo
    predicciones[nombre] = pred
    errores[nombre] = errores_empresa(test[TARGET], pred, test.company_id)
    filas_test.append({'Modelo': nombre, **metricas(test[TARGET], pred, test.company_id)})
resultados_test = pd.DataFrame(filas_test).set_index('Modelo')
display(resultados_test.round(6))

,MAE por empresa,MAE por fila,RMSE por fila,R2 por fila
Modelo,,,,
Base,10.982208,10.986038,13.683007,0.117459
Base + madurez,6.043094,6.042607,7.584809,0.728818
Base + componentes,5.943387,5.926749,7.533025,0.732508
Base + componentes + madurez,5.828083,5.825539,7.315772,0.747715


## 4. Contraste de la hipótesis: diferencias pareadas por empresa

Calculamos la diferencia de errores en cada empresa de prueba. El bootstrap
remuestrea empresas completas (2.000 réplicas), conservando emparejados los errores
de ambos modelos. De esa distribución obtenemos un intervalo percentil del 95%
para la reducción media de MAE.

El intervalo cuantifica variación entre empresas de prueba con estos modelos ya
ajustados. No incluye toda la incertidumbre de reentrenamiento ni garantiza
independencia entre empresas que comparten país o industria. Es una evaluación
exploratoria; el diagnóstico secundario no se presenta como un contraste
confirmatorio adicional ni se usan las cinco particiones como cinco muestras
independientes para calcular un p-valor.

In [6]:
def comparar_pareado(nombre_base, nombre_nuevo, seed=SEED, replicas=2000):
    a, b = errores[nombre_base].align(errores[nombre_nuevo], join='inner')
    assert len(a) == test.company_id.nunique() and a.notna().all() and b.notna().all()
    diferencia = (a - b).to_numpy()
    rng = np.random.default_rng(seed)
    simulaciones = np.empty(replicas)
    for inicio in range(0, replicas, 100):
        n = min(100, replicas - inicio)
        indices = rng.integers(0, len(diferencia), size=(n, len(diferencia)))
        simulaciones[inicio:inicio+n] = diferencia[indices].mean(axis=1)
    lo, hi = np.quantile(simulaciones, [.025, .975])
    return {'Comparación': f'{nombre_base} → {nombre_nuevo}',
            'Reducción MAE (puntos porcentuales)': diferencia.mean(),
            'Reducción relativa (%)': 100 * diferencia.mean() / a.mean(),
            'IC95 inferior': lo, 'IC95 superior': hi,
            '% empresas con menor error': 100 * (diferencia > 0).mean()}

principal = comparar_pareado('Base', 'Base + madurez')
secundaria = comparar_pareado('Base + componentes', 'Base + componentes + madurez')
display(pd.DataFrame([principal, secundaria]).set_index('Comparación').round(6))
tabla_cv = resultados_cv.pivot(index='Partición', columns='Modelo', values='MAE por empresa')
mejoras_cv = tabla_cv['Base'] - tabla_cv['Base + madurez']
print(f'Madurez redujo el MAE en {int(mejoras_cv.gt(0).sum())} de las {len(mejoras_cv)} particiones de desarrollo.')

,Reducción MAE (puntos porcentuales),Reducción relativa (%),IC95 inferior,IC95 superior,% empresas con menor error
Comparación,,,,,
Base → Base + madurez,4.939114,44.973782,4.781420,5.094322,96.706587
Base + componentes → Base + componentes + madurez,0.115303,1.940030,0.085972,0.158475,58.483034


Madurez redujo el MAE en 5 de las 5 particiones de desarrollo.


## 5. Relación con la pregunta original de H2

H2 también pregunta por diferencias entre industrias. Agregar madurez cambia
la comparación: pasamos a comparar sectores a igual madurez además de los otros
controles. Mostramos cómo se modifican los coeficientes sectoriales respecto de
la categoría de referencia. Son asociaciones ajustadas en puntos porcentuales,
no efectos causales ni diferencias cuya significación se haya contrastado aquí.

In [7]:
tablas_coef = {}
for nombre in ['Base', 'Base + madurez']:
    ajustado = ajustes[nombre]
    nombres = ajustado.named_steps['preparar'].get_feature_names_out()
    coef = pd.Series(ajustado.named_steps['regresion'].coef_, index=nombres)
    sectores = coef.loc[coef.index.str.startswith('categoricas__industry_')].copy()
    sectores.index = sectores.index.str.replace('categoricas__industry_', '', regex=False)
    tablas_coef[nombre] = sectores
referencia = ajustes['Base'].named_steps['preparar'].named_transformers_['categoricas'].categories_[0][0]
print(f'Industria de referencia: {referencia}')
display(pd.DataFrame(tablas_coef).round(4))

Industria de referencia: Agriculture


,Base,Base + madurez
Consulting,-0.5293,-0.1359
Education,-0.0268,-0.1060
Finance,3.3313,0.9552
Healthcare,-0.6289,-0.3018
Logistics,0.5514,0.1470
Manufacturing,-0.4288,-0.1878
Retail,-0.3285,-0.1374
Technology,7.4845,0.0197


## 6. Qué permite decidir esta prueba

La decisión requiere dos evaluaciones separadas:

1. **Utilidad:** mejora del error, estabilidad entre particiones y aporte frente
   a los componentes. Un índice puede ser útil frente a controles básicos y
   redundante cuando ya están incluidos sus componentes.
2. **Admisibilidad:** fórmula, momento de medición y disponibilidad real. Mejorar
   la predicción no descarta leakage. SIn la documentacion de como se establecio la variable de *ai_maturity_score* no podemos asegurar el leakage

Para H2, si mejora la predicción, corresponde considerarlo como especificación
alternativa o análisis de sensibilidad. Incorporarlo al modelo principal requiere
resolver su construcción y decidir si controlar por madurez responde a la pregunta
sectorial que queremos contestar. La prueba no modifica las conclusiones del P2.

In [8]:
evidencia = principal['IC95 inferior'] > 0
if evidencia:
    texto = (f"En esta evaluación, los resultados apoyan H₁: agregar madurez reduce el MAE "
             f"por empresa en {principal['Reducción MAE (puntos porcentuales)']:.3f} puntos "
             f"porcentuales ({principal['Reducción relativa (%)']:.1f}%). "
             f"IC bootstrap 95%: [{principal['IC95 inferior']:.3f}, {principal['IC95 superior']:.3f}].")
else:
    texto = ('El intervalo de reducción de MAE no queda completamente por encima de cero. '
             'Esta evaluación no aporta evidencia suficiente a favor de H₁.')
display(Markdown(texto))
display(Markdown('**Decisión:** considerar madurez en una variante de H2 si su mejora resulta útil, '
                 'manteniendo pendiente la validación de su construcción y disponibilidad. '
                 'No incluir automáticamente el índice junto con sus componentes.'))
assert all(sha256(Path(p)) == huella for p, huella in huellas.items())
print('OK: el CSV de entrada no se modificó.')

En esta evaluación, los resultados apoyan H₁: agregar madurez reduce el MAE por empresa en 4.939 puntos porcentuales (45.0%). IC bootstrap 95%: [4.781, 5.094].

**Decisión:** considerar madurez en una variante de H2 si su mejora resulta útil, manteniendo pendiente la validación de su construcción y disponibilidad. No incluir automáticamente el índice junto con sus componentes.

OK: el CSV de entrada no se modificó.
